In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score
import shap

In [2]:
dataset=pd.read_csv('/home/yichuan/ywc/meta-labeling/stock_dataset.csv')
label=pd.read_csv('/home/yichuan/ywc/meta-labeling/stock_label_r10.csv')
dataset

,date,stock,BasicFactor_High,BasicFactor_Low,BasicFactor_Net_Assets_Today,BasicFactor_Net_Cash_Flows_Oper_Act_Ttm,BasicFactor_Open,BasicFactor_S_Dq_Turn,BasicFactor_S_Fa_Assetsturn,BasicFactor_S_Fa_Debttoassets,...,BasicFactor_S_Fa_Grossprofitmargin,BasicFactor_S_Fa_Ocfps,BasicFactor_S_Fa_Roe,BasicFactor_S_Val_Pb_New,BasicFactor_S_Val_Pe,BasicFactor_S_Val_Ps,BasicFactor_Volume,ret20_final,turn20_final,vol20_final
0,2000-01-04,000001.SZ,-1.876617,-1.983328,9.405113e+08,7.770691e+08,-1.875596,-0.983423,NaN,NaN,...,NaN,NaN,NaN,-0.414404,265.713161,2.628491,43427.606865,-0.069360,-0.058098,-0.069360
1,2000-01-04,000002.SZ,-5.179042,-5.108496,2.844510e+08,-3.327838e+08,-5.074491,0.034307,NaN,NaN,...,NaN,NaN,NaN,-3.867642,74.556222,-8.043314,23101.339587,0.103493,0.062950,0.103493
2,2000-01-04,000003.SZ,-6.838942,-6.640623,-6.059850e+08,-7.849902e+06,-6.710954,-0.238048,NaN,NaN,...,NaN,NaN,NaN,-1.895098,NaN,-6.130668,6018.767095,0.120330,-0.134924,0.120330
3,2000-01-04,000004.SZ,-0.966288,-0.936544,-2.807509e+08,1.240277e+07,-0.880764,0.544817,NaN,NaN,...,NaN,NaN,NaN,5.170559,NaN,-2.917421,3388.166723,1.195657,1.115978,1.195657
4,2000-01-04,000005.SZ,-7.308703,-6.861967,-2.810491e+08,-1.842273e+08,-6.917121,-0.951641,NaN,NaN,...,NaN,NaN,NaN,-2.953761,-6.898533,67.739555,-7642.097263,-0.511783,-0.499846,-0.511783
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33034269,2024-12-31,688800.SH,31.874940,27.467773,-2.021310e+10,-5.534121e+09,31.204712,4.644974,NaN,NaN,...,NaN,NaN,NaN,0.355742,-18.607485,-6.882643,-158304.171613,0.010856,5.037429,0.040634
33034270,2024-12-31,688819.SH,-2.447078,-2.078618,-3.812556e+10,-1.291345e+10,-2.302136,-2.036765,NaN,NaN,...,NaN,NaN,NaN,-1.824081,-58.758740,-15.506271,-441778.791361,-0.001480,-2.067374,-0.009256
33034271,2024-12-31,688981.SH,57.334973,53.813284,3.877383e+10,-1.182994e+10,57.189802,3.553902,NaN,NaN,...,NaN,NaN,NaN,1.839002,96.648163,-6.018871,121137.693417,0.003565,3.976764,0.013913
33034272,2024-12-31,689009.SH,18.104530,18.161132,-4.759035e+10,-1.176733e+10,18.178853,-0.836783,NaN,NaN,...,NaN,NaN,NaN,2.130875,-13.350178,-12.682466,-382065.925491,0.005175,-0.727440,-0.003912


In [7]:
def dateindex(df):
    df.rename(columns={df.columns[0]:'date'},inplace=True)
    try:
        df['date'] = pd.to_datetime(df['date'], format='%Y%m%d')
    except:
        df['date'] = pd.to_datetime(df['date'])
    df.set_index('date', inplace=True)
    return df

In [12]:
close = pd.read_feather('/home/yichuan/ywc/preparation/BasicFactor_Close.txt')
close = dateindex(close)
close=close[[col for col in close.columns if not col.endswith('BJ')]]
close=close.loc[(close.index >= pd.Timestamp('2000-01-01')) & (close.index <= pd.Timestamp('2024-12-31'))]
close.to_parquet('/home/yichuan/ywc/meta-labeling/close.parquet', index=True, engine='fastparquet')
close = pd.read_parquet('/home/yichuan/ywc/meta-labeling/close.parquet')
close

,000001.SZ,000002.SZ,000003.SZ,000004.SZ,000005.SZ,000006.SZ,000007.SZ,000008.SZ,000009.SZ,000010.SZ,...,688755.SH,301595.SZ,603014.SH,001390.SZ,301590.SZ,603049.SH,688775.SH,603382.SH,301678.SZ,603400.SH
date,,,,,,,,,,,,,,,,,,,,,
2000-01-04,18.29,10.30,5.74,8.74,6.24,9.70,8.28,44.91,3.59,7.47,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-05,18.06,10.04,5.68,9.02,6.25,9.61,8.25,45.65,3.60,7.58,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-06,18.78,10.51,5.85,9.40,6.47,9.90,8.48,46.78,3.73,7.78,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-07,19.54,10.99,6.12,9.87,6.76,10.45,9.33,48.95,3.81,8.10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-10,20.14,11.44,6.36,10.36,7.01,10.82,9.61,52.47,3.86,8.32,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-25,11.92,7.54,NaN,14.63,NaN,7.73,6.63,3.20,9.35,2.73,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-12-26,11.86,7.51,NaN,15.53,NaN,7.74,6.80,3.09,9.28,2.77,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-12-27,11.83,7.55,NaN,14.48,NaN,8.02,7.10,3.10,9.42,2.80,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
label

,date,level_1,future_ret
0,2000-01-04,000001.SZ,-0.033355
1,2000-01-04,000002.SZ,-0.029559
2,2000-01-04,000003.SZ,0.008673
3,2000-01-04,000004.SZ,0.230894
4,2000-01-04,000005.SZ,0.167793
...,...,...,...
33034269,2024-12-31,603049.SH,NaN
33034270,2024-12-31,688775.SH,NaN
33034271,2024-12-31,603382.SH,NaN
33034272,2024-12-31,301678.SZ,NaN


In [4]:
def check_keys_with_details(df1, df2):
    """检查主键并显示差异"""
    keys1 = set(zip(df1.iloc[:, 0], df1.iloc[:, 1]))
    keys2 = set(zip(df2.iloc[:, 0], df2.iloc[:, 1]))
    
    only_in_1 = keys1 - keys2
    only_in_2 = keys2 - keys1
    
    is_same = keys1 == keys2
    
    print(f"表1主键数量: {len(keys1)}")
    print(f"表2主键数量: {len(keys2)}")
    print(f"主键是否相同: {is_same}")
    
    if only_in_1:
        print(f"仅在表1中的主键: {only_in_1}")
    if only_in_2:
        print(f"仅在表2中的主键: {only_in_2}")
    
    return is_same

# 使用示例
check_keys_with_details(dataset, label)

表1主键数量: 33034274
表2主键数量: 33034274
主键是否相同: True


True

In [5]:
label_aligned = label.set_index([label.columns[0], label.columns[1]]).loc[
    list(zip(dataset.iloc[:, 0], dataset.iloc[:, 1]))
].reset_index()

In [6]:
label_aligned

,date,level_1,future_ret
0,2000-01-04,000001.SZ,-0.033355
1,2000-01-04,000002.SZ,-0.029559
2,2000-01-04,000003.SZ,0.008673
3,2000-01-04,000004.SZ,0.230894
4,2000-01-04,000005.SZ,0.167793
...,...,...,...
33034269,2024-12-31,688800.SH,0.270117
33034270,2024-12-31,688819.SH,-0.045855
33034271,2024-12-31,688981.SH,0.050791
33034272,2024-12-31,689009.SH,0.025360


In [7]:
label=label_aligned.copy()
result=(dataset.iloc[:, :2].values == label.iloc[:, :2].values).all()
print(result)
label = label.rename(columns={label.columns[1]: 'stock'})
label

True


,date,stock,future_ret
0,2000-01-04,000001.SZ,-0.033355
1,2000-01-04,000002.SZ,-0.029559
2,2000-01-04,000003.SZ,0.008673
3,2000-01-04,000004.SZ,0.230894
4,2000-01-04,000005.SZ,0.167793
...,...,...,...
33034269,2024-12-31,688800.SH,0.270117
33034270,2024-12-31,688819.SH,-0.045855
33034271,2024-12-31,688981.SH,0.050791
33034272,2024-12-31,689009.SH,0.025360


In [8]:
condition = label.iloc[:, 2].notna()
# 同时过滤A表和B表
X_data = dataset[condition]
y_data = label[condition]
X_data

,date,stock,BasicFactor_High,BasicFactor_Low,BasicFactor_Net_Assets_Today,BasicFactor_Net_Cash_Flows_Oper_Act_Ttm,BasicFactor_Open,BasicFactor_S_Dq_Turn,BasicFactor_S_Fa_Assetsturn,BasicFactor_S_Fa_Debttoassets,...,BasicFactor_S_Fa_Grossprofitmargin,BasicFactor_S_Fa_Ocfps,BasicFactor_S_Fa_Roe,BasicFactor_S_Val_Pb_New,BasicFactor_S_Val_Pe,BasicFactor_S_Val_Ps,BasicFactor_Volume,ret20_final,turn20_final,vol20_final
0,2000-01-04,000001.SZ,-1.876617,-1.983328,9.405113e+08,7.770691e+08,-1.875596,-0.983423,NaN,NaN,...,NaN,NaN,NaN,-0.414404,265.713161,2.628491,43427.606865,-0.069360,-0.058098,-0.069360
1,2000-01-04,000002.SZ,-5.179042,-5.108496,2.844510e+08,-3.327838e+08,-5.074491,0.034307,NaN,NaN,...,NaN,NaN,NaN,-3.867642,74.556222,-8.043314,23101.339587,0.103493,0.062950,0.103493
2,2000-01-04,000003.SZ,-6.838942,-6.640623,-6.059850e+08,-7.849902e+06,-6.710954,-0.238048,NaN,NaN,...,NaN,NaN,NaN,-1.895098,NaN,-6.130668,6018.767095,0.120330,-0.134924,0.120330
3,2000-01-04,000004.SZ,-0.966288,-0.936544,-2.807509e+08,1.240277e+07,-0.880764,0.544817,NaN,NaN,...,NaN,NaN,NaN,5.170559,NaN,-2.917421,3388.166723,1.195657,1.115978,1.195657
4,2000-01-04,000005.SZ,-7.308703,-6.861967,-2.810491e+08,-1.842273e+08,-6.917121,-0.951641,NaN,NaN,...,NaN,NaN,NaN,-2.953761,-6.898533,67.739555,-7642.097263,-0.511783,-0.499846,-0.511783
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33034268,2024-12-31,688799.SH,24.124183,23.660234,-1.884276e+09,-1.377558e+08,23.881717,-2.442629,NaN,NaN,...,NaN,NaN,NaN,-1.409886,-61.778969,-6.866235,-164305.611278,0.004923,-3.037957,-0.017149
33034269,2024-12-31,688800.SH,31.874940,27.467773,-2.021310e+10,-5.534121e+09,31.204712,4.644974,NaN,NaN,...,NaN,NaN,NaN,0.355742,-18.607485,-6.882643,-158304.171613,0.010856,5.037429,0.040634
33034270,2024-12-31,688819.SH,-2.447078,-2.078618,-3.812556e+10,-1.291345e+10,-2.302136,-2.036765,NaN,NaN,...,NaN,NaN,NaN,-1.824081,-58.758740,-15.506271,-441778.791361,-0.001480,-2.067374,-0.009256
33034271,2024-12-31,688981.SH,57.334973,53.813284,3.877383e+10,-1.182994e+10,57.189802,3.553902,NaN,NaN,...,NaN,NaN,NaN,1.839002,96.648163,-6.018871,121137.693417,0.003565,3.976764,0.013913


In [9]:
y_data

,date,stock,future_ret
0,2000-01-04,000001.SZ,-0.033355
1,2000-01-04,000002.SZ,-0.029559
2,2000-01-04,000003.SZ,0.008673
3,2000-01-04,000004.SZ,0.230894
4,2000-01-04,000005.SZ,0.167793
...,...,...,...
33034268,2024-12-31,688799.SH,-0.005507
33034269,2024-12-31,688800.SH,0.270117
33034270,2024-12-31,688819.SH,-0.045855
33034271,2024-12-31,688981.SH,0.050791


In [10]:
def dateindex(df):
    df.rename(columns={df.columns[0]:'date'},inplace=True)
    try:
        df['date'] = pd.to_datetime(df['date'], format='%Y%m%d')
    except:
        df['date'] = pd.to_datetime(df['date'])
    df.set_index('date', inplace=True)
    return df

In [11]:
y_data.iloc[:,:2].to_csv('/home/yichuan/ywc/meta-labeling/date_stock_index.csv', index=True)

In [12]:
date_stock_index=pd.read_csv('/home/yichuan/ywc/meta-labeling/date_stock_index.csv')
date_stock_index

,Unnamed: 0,date,stock
0,0,2000-01-04,000001.SZ
1,1,2000-01-04,000002.SZ
2,2,2000-01-04,000003.SZ
3,3,2000-01-04,000004.SZ
4,4,2000-01-04,000005.SZ
...,...,...,...
15421311,33034268,2024-12-31,688799.SH
15421312,33034269,2024-12-31,688800.SH
15421313,33034270,2024-12-31,688819.SH
15421314,33034271,2024-12-31,688981.SH


In [13]:
import xgboost as xgb
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import mean_squared_error, r2_score

# ========= 数据准备 =========
# 保留 date 和 stock
dates = X_data.iloc[:, 0]
stocks = X_data.iloc[:, 1]

X_features = X_data.iloc[:, 2:]
y_labels = y_data.iloc[:, 2]

# 所有唯一交易日（必须是已排序的）
unique_dates = np.sort(dates.unique())

train_days = 500
test_days = 20
step_days = 20

n_iter = (len(unique_dates) - train_days - test_days) // step_days + 1
print(unique_dates,len(unique_dates),n_iter)

results = []

with tqdm(total=n_iter, desc="Walk-forward validation", ncols=100) as pbar:
    for i in range(n_iter):
        # ===== 确定时间窗口 =====
        train_date_start = i * step_days
        train_date_end = train_date_start + train_days
        test_date_end = train_date_end + test_days

        train_dates = unique_dates[train_date_start:train_date_end]
        test_dates = unique_dates[train_date_end:test_date_end]

        # Use an internal validation set for early stopping. Because stock_label_r10
        # is a 10-day forward-return label, leave a 10-day embargo on both sides of
        # validation so that neither fitting nor validation labels overlap the next
        # time segment. The test set is used only once, for final prediction.
        validation_days = 20
        label_horizon_days = 10
        fit_dates = train_dates[:-(validation_days + 2 * label_horizon_days)]
        validation_dates = train_dates[
            -(validation_days + label_horizon_days):-label_horizon_days
        ]

        # ===== �������չ������� =====
        fit_mask = dates.isin(fit_dates)
        validation_mask = dates.isin(validation_dates)
        test_mask = dates.isin(test_dates)

        X_fit = X_features.loc[fit_mask]
        y_fit = y_labels.loc[fit_mask]

        X_validation = X_features.loc[validation_mask]
        y_validation = y_labels.loc[validation_mask]

        X_test = X_features.loc[test_mask]
        y_test = y_labels.loc[test_mask]

        # ===== XGBoost ���ݸ�ʽ =====
        dfit = xgb.DMatrix(X_fit, label=y_fit)
        dvalidation = xgb.DMatrix(X_validation, label=y_validation)
        dtest = xgb.DMatrix(X_test, label=y_test)

        evals = [(dfit, "train"), (dvalidation, "validation")]

        params = {
            "objective": "reg:squarederror",
            "eta": 0.05,
            "max_depth": 6,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "seed": 42,
            "nthread": -1
        }

        model = xgb.train(
            params,
            dfit,
            num_boost_round=300,
            evals=evals,
            verbose_eval=False,
            callbacks=[xgb.callback.EarlyStopping(rounds=30, save_best=True)]
        )

        # ===== ���Լ�Ԥ�� =====
        y_pred = model.predict(dtest)

        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)

        df_result = pd.DataFrame({
            "date": dates.loc[test_mask].values,
            "stock": stocks.loc[test_mask].values,
            "true": y_test.values,
            "pred": y_pred,
            "rmse": rmse,
            "r2": r2
        })

        results.append(df_result)

        pbar.update(1)

final_results = pd.concat(results, ignore_index=True)
print(final_results.shape)
print(final_results.head())


['2000-01-04' '2000-01-05' '2000-01-06' ... '2024-12-27' '2024-12-30'
 '2024-12-31'] 6058 277


Walk-forward validation: 100%|████████████████████████████████████| 277/277 [10:11<00:00,  2.21s/it]


(14809725, 6)
         date      stock      true      pred      rmse        r2
0  2002-02-04  000001.SZ -0.020068 -0.001514  0.085806 -0.737197
1  2002-02-04  000002.SZ  0.001590 -0.001107  0.085806 -0.737197
2  2002-02-04  000004.SZ  0.071258 -0.001107  0.085806 -0.737197
3  2002-02-04  000005.SZ  0.098074 -0.001107  0.085806 -0.737197
4  2002-02-04  000006.SZ  0.014334 -0.001107  0.085806 -0.737197


In [14]:
final_results.to_csv('/home/yichuan/ywc/meta-labeling/primary_signal.csv', index=False)

In [15]:
sig=pd.read_csv('/home/yichuan/ywc/meta-labeling/primary_signal.csv')
sig

,date,stock,true,pred,rmse,r2
0,2002-02-04,000001.SZ,-0.020068,-0.001514,0.085806,-0.737197
1,2002-02-04,000002.SZ,0.001590,-0.001107,0.085806,-0.737197
2,2002-02-04,000004.SZ,0.071258,-0.001107,0.085806,-0.737197
3,2002-02-04,000005.SZ,0.098074,-0.001107,0.085806,-0.737197
4,2002-02-04,000006.SZ,0.014334,-0.001107,0.085806,-0.737197
...,...,...,...,...,...,...
14809720,2024-12-05,688799.SH,0.010264,-0.008326,0.100124,-0.064301
14809721,2024-12-05,688800.SH,0.146541,-0.010240,0.100124,-0.064301
14809722,2024-12-05,688819.SH,-0.012078,-0.008736,0.100124,-0.064301
14809723,2024-12-05,688981.SH,-0.018094,-0.010403,0.100124,-0.064301
